# Colab Training Launcher

This notebook launches the package training scripts. It does not contain model or training logic.

Expected Drive dataset path:

```text
/content/drive/MyDrive/VOC_Dataset/voc2012_processed/
```


In [ ]:
import yaml
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
REPO_URL = 'https://github.com/lovisticsdev/multilabel_image_classifier_web.git'

%cd /content
!rm -rf app
!git clone $REPO_URL app
%cd /content/app
!git log -1 --oneline


/content
Cloning into 'app'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 108 (delta 2), reused 108 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 41.43 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/app
95dea80 (HEAD -> main, origin/main, origin/HEAD) chore: prepare multilabel image classifier project


In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!pip install -e '.[dev,train]'


In [ ]:
config_path = Path('configs/voc20.yaml')

dataset_root = Path('/content/drive/MyDrive/VOC_Dataset/voc2012_processed')

paths_to_check = {
    'train_images': dataset_root / 'train' / 'images',
    'val_images': dataset_root / 'val' / 'images',
    'train_annotations': dataset_root / 'train_annotations.csv',
    'val_annotations': dataset_root / 'val_annotations.csv',
}

for name, path in paths_to_check.items():
    if not path.exists():
        raise FileNotFoundError(f'{name} not found: {path}')

cfg = yaml.safe_load(config_path.read_text())

cfg['dataset']['train_images'] = str(paths_to_check['train_images'])
cfg['dataset']['val_images'] = str(paths_to_check['val_images'])
cfg['dataset']['train_annotations'] = str(paths_to_check['train_annotations'])
cfg['dataset']['val_annotations'] = str(paths_to_check['val_annotations'])

cfg.setdefault('outputs', {})
cfg['outputs']['checkpoint_dir'] = '/content/drive/MyDrive/VOC_Dataset/checkpoints'
cfg['outputs']['export_dir'] = '/content/drive/MyDrive/VOC_Dataset/exported_model'
cfg['outputs']['report_dir'] = '/content/drive/MyDrive/VOC_Dataset/model_outputs'

config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(config_path.read_text())


project_name: multilabel-image-classifier-web
dataset:
  train_images: /content/drive/MyDrive/VOC_Dataset/voc2012_processed/train/images
  val_images: /content/drive/MyDrive/VOC_Dataset/voc2012_processed/val/images
  train_annotations: /content/drive/MyDrive/VOC_Dataset/voc2012_processed/train_annotations.csv
  val_annotations: /content/drive/MyDrive/VOC_Dataset/voc2012_processed/val_annotations.csv
  image_name_column: image_name
  class_mode: all
model:
  architecture: efficientnet_b2
  pretrained: true
  image_size: 256
  dropout: 0.3
training:
  batch_size: 32
  epochs: 30
  learning_rate: 0.0003
  weight_decay: 0.01
  early_stopping_patience: 7
  num_workers: 2
  seed: 42
  mixed_precision: auto
  device: auto
  compute_pos_weight: true
thresholds:
  tune: true
  metric: f1
  default_threshold: 0.5
  search_min: 0.05
  search_max: 0.95
  search_steps: 19
outputs:
  checkpoint_dir: /content/drive/MyDrive/VOC_Dataset/checkpoints
  export_dir: /content/drive/MyDrive/VOC_Dataset/expor

In [ ]:
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Training will run on CPU and may be slow.')


torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [ ]:
!python scripts/inspect_dataset.py --config configs/voc20.yaml



train: 5717 rows
person         2142
chair           656
dog             636
car             621
cat             540
bottle          399
bird            399
sofa            359
aeroplane       328
diningtable     318
tvmonitor       299
pottedplant     289
bicycle         281
train           275
motorbike       274
boat            264
horse           238
bus             219
sheep           171
cow             155
dtype: int64

val: 5823 rows
person         2232
dog             661
chair           642
car             608
cat             544
bird            374
bottle          369
aeroplane       348
sofa            336
diningtable     323
tvmonitor       296
bicycle         290
pottedplant     279
train           275
motorbike       262
boat            252
horse           245
bus             211
sheep           155
cow             154
dtype: int64


In [8]:
!python scripts/train.py --config configs/voc20.yaml


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth
100% 35.2M/35.2M [00:00<00:00, 127MB/s]
Epoch 1/30: 100% 179/179 [20:19<00:00,  6.81s/it, loss=0.602]
epoch=1 train_loss=0.6019 macro_f1=0.7819
Epoch 2/30: 100% 179/179 [01:42<00:00,  1.75it/s, loss=0.332]
epoch=2 train_loss=0.3320 macro_f1=0.8013
Epoch 3/30: 100% 179/179 [01:42<00:00,  1.74it/s, loss=0.25]
epoch=3 train_loss=0.2501 macro_f1=0.7983
Epoch 4/30: 100% 179/179 [01:41<00:00,  1.76it/s, loss=0.211]
epoch=4 train_loss=0.2109 macro_f1=0.8011
Epoch 5/30: 100% 179/179 [01:45<00:00,  1.69it/s, loss=0.177]
epoch=5 train_loss=0.1774 macro_f1=0.8066
Epoch 6/30: 100% 179/179 [01:45<00:00,  1.70it/s, loss=0.155]
epoch=6 train_loss=0.1546 macro_f1=0.8030
Epoch 7/30: 100% 179/179 [01:45<00:00,  1.70it/s, loss=0.137]
epoch=7 train_loss=0.1374 macro_f1=0.8061
Epoch 8/30: 100% 179/179 [01:46<00:00,  1.68it/s, loss=0.12]
epoch

In [9]:
!python scripts/evaluate.py --config configs/voc20.yaml --artifact-dir /content/drive/MyDrive/VOC_Dataset/exported_model


artifact_dir: /content/drive/MyDrive/VOC_Dataset/exported_model
macro_f1: 0.8255
micro_f1: 0.8362

Per-class metrics:
aeroplane    precision=0.9373 recall=0.9454 f1=0.9413 threshold=0.55
bicycle      precision=0.9303 recall=0.7828 f1=0.8502 threshold=0.95
bird         precision=0.9347 recall=0.8797 f1=0.9063 threshold=0.90
boat         precision=0.8755 recall=0.8095 f1=0.8412 threshold=0.95
bottle       precision=0.6319 recall=0.5908 f1=0.6106 threshold=0.80
bus          precision=0.9430 recall=0.8626 f1=0.9010 threshold=0.95
car          precision=0.7929 recall=0.7681 f1=0.7803 threshold=0.60
cat          precision=0.9319 recall=0.9301 f1=0.9310 threshold=0.85
chair        precision=0.6435 recall=0.7508 f1=0.6930 threshold=0.65
cow          precision=0.8205 recall=0.8312 f1=0.8258 threshold=0.95
diningtable  precision=0.8327 recall=0.7090 f1=0.7659 threshold=0.95
dog          precision=0.8986 recall=0.8714 f1=0.8848 threshold=0.85
horse        precision=0.9333 recall=0.8571 f1=0.8936 